# QELM on the measurement record M: TFIM, Haar and Clifford dynamics

Issue #171. The experiment of `03_QELM_M_alpha_ratio.ipynb` (`experiments/QELM_M/exp_QELM_M_ratio.py`), run with three unitary dynamics on the system qubits instead of one:

- `TFIM`: the disordered transverse-field Ising model, a new disorder per sample, the baseline;
- `haar_circuit`: a brickwork circuit of Haar-random two-qubit gates, drawn from U(4) (`QELM.models.circuits.HaarBrickworkCircuit`), no conserved quantity;
- `clifford_circuit`: the same brickwork with two-qubit Clifford gates, drawn uniformly from the 11 520 elements of the group (`QELM.models.circuits.CliffordBrickworkCircuit`).

Everything else is the TFIM run: the ancilla `R` entangled with the charge of the system, the system qubits measured in Z with probability `p` after each of the `nbr_time_evolution` steps, the record `M` as features (0 for an unmeasured qubit), the measured label of `R` (the initial charge sector, ±1) as target, a ridge readout with `n_params = nbr_time_evolution * N + 1` weights and `n_train = alpha_ratio * n_params` training samples. For a circuit, one time-evolution step is `layers_per_step` brickwork layers (1 by default, each a full period: gates on the even bonds, then on the odd ones), and every sample gets new gates, as every TFIM sample gets a new disorder (`fixed_instance: false`, see `conf/models/haar_circuit.yaml`).

The sweep is the grid of notebook 03, `p` and `alpha_ratio`, once per model. The figures:

1. the metrics against the ratio, one row of panels per model and `N`, to locate the double-descent peak at `alpha_ratio = 1`;
2. the heatmaps over `p` and the ratio, one row per model and `N` (issue #164);
3. the comparison of the three models on the same axes: the balanced accuracy of the readout against `p` at the largest ratio, and the NMSE against the ratio at `p = 1`.

The Bayes accuracy `sign(<Z_0>)` (`accuracy_optimal`) is no longer drawn (issue #195): it used the exact state of `R`, `P(Y | rho_R)`, which knows the unitary of each sample, while the readout only sees the record `M`. A ceiling for the readout would need `P(Y | M)`.

**Qubit counts (issue #235).** `N` is the number of system qubits, the ancilla excluded; the simulated state has `N_total = N + 1` qubits. The sizes of this notebook, its defaults and the results quoted in its text were moved to this convention: a result obtained at `N = 6` before issue #235, when `N` counted the ancilla, is quoted here at `N = 5`, the same system.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_m_circuits_notebook.py` does for a small sweep). `MODELS` lists the dynamics to compare, the `models` values of `conf/models/`. The other parameters are those of notebook 03. Each sweep goes to this notebook's own folder, `outputs/QELM_M_circuits/<date_time>/`, with one subfolder per model; `RUN = False` reads back the latest sweep of this folder, and `RESUME` set to the path of an earlier sweep folder completes it (the runs already done are skipped, issue #168).

The default grid has `11 x 12` points per model, seed and `N`, so `3 x 3 x 132 = 1188` runs at `N = 3`, a few seconds each. `FIXED = True` adds the sweep of the last section, the circuits with a fixed draw of their gates: `2 x 3 x 132 = 792` more runs.

In [ ]:
# Unitary dynamics to compare, models values of conf/models/.
MODELS = os.environ.get("MODELS", "TFIM haar_circuit clifford_circuit")
N_VALUES = os.environ.get("N_VALUES", "3")  # system qubits N, ancilla excluded
SEEDS = os.environ.get("SEEDS", "42 43 44")
# Measurement probability, 0 to 1 (the y axis of the heatmaps).
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0")
# The values of p drawn as curves against the ratio.
CURVE_P_VALUES = os.environ.get("CURVE_P_VALUES", "0.2 0.5 1.0")
# Training samples per weight, n_train / n_params (experiment.alpha_ratio),
# 0.25 to 10 on a log scale.
ALPHA_RATIOS = os.environ.get(
    "ALPHA_RATIOS", "0.25 0.35 0.5 0.71 1.0 1.41 2.0 2.83 4.0 5.66 8.0 10.0"
)
N_TEST = os.environ.get("N_TEST", "200")  # test samples per run
ALPHA = os.environ.get("ALPHA", "1e-3")  # ridge penalty (training.alpha)
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"
# Sweep folder to complete (runs already done are skipped); "" for a new sweep.
RESUME = os.environ.get("RESUME", "")
# Also sweep each circuit with one fixed draw of its gates (last section).
FIXED = os.environ.get("FIXED", "1") == "1"

## Launch the sweeps

One sweep of `experiments/QELM_M/exp_QELM_M_ratio.sh` per model, with `MODEL` set, each in its own subfolder of the sweep folder.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "QELM_M_circuits"
models = MODELS.split()
n_runs = (
    len(N_VALUES.split())
    * len(SEEDS.split())
    * len(P_VALUES.split())
    * len(ALPHA_RATIOS.split())
)


def launch(model, out_dir, extra_overrides=""):
    """Run the ratio sweep of one model into out_dir, one bar for the sweep."""
    env = {
        **os.environ,
        "MODEL": model,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "ALPHA_RATIOS": ALPHA_RATIOS,
        "N_TEST": N_TEST,
        "ALPHA": ALPHA,
        "OUT_DIR": str(out_dir),
        "EXTRA_OVERRIDES": extra_overrides,
    }
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M_ratio.sh"], total=n_runs, env=env
    )
    failed = out_dir / "failed_runs.txt"
    if failed.is_file():
        print(
            f"WARNING: failed runs, missing from the figures:\n{failed.read_text()}"
            f'Set RESUME = "{sweep_dir}" and run again to retry them.'
        )


if RUN:
    sweep_dir = Path(RESUME) if RESUME else new_sweep_dir(OUT_ROOT, EXPERIMENT)
    for model in models:
        print(f"model: {model}")
        launch(model, sweep_dir / model)
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

One table per model, the mean over the seeds at each `(N, p, ratio)`, with the ratio actually used, `n_train / n_features`, from each run's `metrics.json`. The model of each run is read back from its config and checked against its folder.

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label, log scale).
METRICS = [
    ("nmse", "NMSE (test)", True),
    ("r2", "R2 (test)", False),
    ("balanced_accuracy", "balanced accuracy (test)", False),
]
SUMMARY = [k for k, _, _ in METRICS] + ["accuracy", "S_vN_mean"]


def read_table(model, folder, fixed=False):
    """Mean over the seeds of the runs of one model, one row per (N, p, ratio).

    The model and, for a circuit, fixed_instance are read back from each
    run's config and checked.
    """
    keys = [("N", "N"), ("seed", "seed"), ("p", "projective_measurement.p")]
    keys += [("model", "models"), ("fixed", f"{model}.fixed_instance")]
    rows = load_sweep(folder, keys=keys)
    assert {r["model"] for r in rows} == {model}, folder
    if model != "TFIM":
        assert {r["fixed"] for r in rows} == {fixed}, folder
    for r in rows:
        r["ratio"] = r["n_train"] / r["n_features"]
    table = summarize(rows, by=("N", "p", "ratio"), metrics=SUMMARY)
    print(f"{model}: {len(rows)} runs, {len(table)} grid points")
    return table


tables = {model: read_table(model, sweep_dir / model) for model in models}

## Metrics against the ratio

One row of panels per model and `N`, one panel per metric, one curve per `p` of `CURVE_P_VALUES`, the mean over the seeds with the standard error. The dashed line marks the interpolation threshold `alpha_ratio = 1`; on the balanced accuracy, the dotted gray line marks chance, 0.5. Saved as `metrics_vs_alpha_ratio.png` and `.json` in the sweep folder.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

from QELM.utils.plots import (
    color_map,
    draw_curve,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
)

n_values = sizes(tables[models[0]])


def ratio_ticks(ax, ratios):
    """Label the log ratio axis at the powers of 2 within the swept ratios."""
    ticks = [2.0**k for k in range(-6, 7) if min(ratios) <= 2.0**k <= max(ratios)]
    ax.set_xticks(ticks, labels=[f"{t:g}" for t in ticks])
    ax.xaxis.set_minor_formatter(NullFormatter())


def draw_ratio_curves(tables):
    """Metrics against the ratio, one row of panels per model and N."""
    panel_rows = [(model, N) for model in tables for N in n_values]
    fig, axes = subplots(
        len(panel_rows),
        len(METRICS),
        figsize=(6.5 * len(METRICS), 5.0 * len(panel_rows)),
        squeeze=False,
    )
    for row, (model, N) in zip(axes, panel_rows, strict=True):
        table = tables[model]
        curve_ps = [float(p) for p in CURVE_P_VALUES.split()]
        ps = sorted({r["p"] for r in table if r["p"] in curve_ps})
        if not ps:  # none of CURVE_P_VALUES was swept: draw every p
            ps = sorted({r["p"] for r in table})
        colors = color_map([f"p = {p:g}" for p in ps])
        for p in ps:
            rows_p = [r for r in table if r["p"] == p]
            label = f"p = {p:g}"
            for (key, _, log), ax in zip(METRICS, row, strict=True):
                curve = sweep_curve(rows_p, N, "ratio", key)
                draw_curve(ax, curve, colors[label], label=label, log=log)
        for (key, label, _), ax in zip(METRICS, row, strict=True):
            ax.axvline(
                1.0, color="black", linestyle="--", label="interpolation threshold"
            )
            if key == "balanced_accuracy":
                ax.axhline(0.5, color="gray", linestyle=":", label="chance")
            ax.set_xscale("log")
            ratio_ticks(ax, [r["ratio"] for r in table if r["N"] == N])
            ax.set(
                xlabel="alpha_ratio = n_train / n_params",
                ylabel=label,
                title=f"{label}, {model}, N = {N}",
            )
            ax.legend()
    return fig, axes


fig, axes = draw_ratio_curves(tables)
fig.suptitle("QELM on M: metrics against the sample-to-parameter ratio, per dynamics")
figure_path = save_figure(
    fig, sweep_dir, "metrics_vs_alpha_ratio", data={"tables": tables}
)
print(f"figure: {figure_path}")
plt.show()

## Heatmaps over p and the ratio

The heatmaps of notebook 03 (issues #164 and #167), one row per model and `N`: the mean over the seeds at each grid point, `alpha_ratio` on a logarithmic x axis and `p` on the y axis, the dashed white line at `alpha_ratio = 1`. Color scales: NMSE logarithmic, R2 from -1 to 1, accuracies from 0.5 to 1. Saved as `heatmaps_p_alpha_ratio.png` and `.json`.

In [ ]:
from QELM.utils.plots import draw_heatmap, sweep_grid

# (key in metrics.json, label, log color scale, bottom and top of the scale).
HEATMAPS = [
    ("nmse", "NMSE (test)", True, None, None),
    ("r2", "R2 (test)", False, -1.0, 1.0),
    ("accuracy", "accuracy (test)", False, 0.5, 1.0),
    ("balanced_accuracy", "balanced accuracy (test)", False, 0.5, 1.0),
]


def draw_heatmaps(tables):
    """Heatmaps over (ratio, p), one row of panels per model and N.

    Returns the figure, its axes and the grids drawn, by (model, N, metric).
    """
    panel_rows = [(model, N) for model in tables for N in n_values]
    fig, axes = subplots(
        len(panel_rows),
        len(HEATMAPS),
        figsize=(5.5 * len(HEATMAPS), 4.6 * len(panel_rows)),
        squeeze=False,
    )
    grids = {}
    for row, (model, N) in zip(axes, panel_rows, strict=True):
        for (key, label, log_color, vmin, vmax), ax in zip(HEATMAPS, row, strict=True):
            grid = sweep_grid(tables[model], "ratio", "p", key, size=N)
            grids[(model, N, key)] = grid
            draw_heatmap(
                ax,
                grid,
                label=label,
                log_x=True,
                log_color=log_color,
                vmin=vmin,
                vmax=vmax,
            )
            ratio_ticks(ax, grid.x)
            ax.axvline(
                1.0, color="white", linestyle="--", label="interpolation threshold"
            )
            ax.set(
                xlabel="alpha_ratio = n_train / n_params",
                ylabel="measurement probability p",
                title=f"{label}, {model}, N = {N}",
            )
    return fig, axes, grids


fig_maps, axes_maps, grids = draw_heatmaps(tables)
fig_maps.suptitle("QELM on M: metrics over p and the sample-to-parameter ratio")
heatmap_path = save_figure(
    fig_maps, sweep_dir, "heatmaps_p_alpha_ratio", data={"tables": tables}
)
print(f"figure: {heatmap_path}")
plt.show()

## The three dynamics compared

One row of panels per `N`, one curve per model:

- balanced accuracy of the readout against `p`, at the largest ratio of the sweep (the end of the double descent, where finite sampling matters least);
- NMSE against the ratio at the largest `p` of the sweep, to compare the double-descent peaks.

Saved as `comparison_models.png` and `.json`.

In [ ]:
model_colors = color_map(models)
COMPARISON = [
    ("balanced_accuracy", "balanced accuracy (test)"),
    ("nmse", "NMSE (test)"),
]


def draw_comparison(curves):
    """The two comparison panels per N, one curve per (label, table, color,
    linestyle) of curves. Returns the figure, its axes and the data drawn."""
    fig, axes = subplots(
        len(n_values),
        len(COMPARISON),
        figsize=(6.5 * len(COMPARISON), 5.0 * len(n_values)),
        squeeze=False,
    )
    data = {}
    for row, N in zip(axes, n_values, strict=True):
        ax_bal, ax_nmse = row
        for label, table, color, linestyle in curves:
            table = [r for r in table if r["N"] == N]
            ratio_max = max(r["ratio"] for r in table)
            p_max = max(r["p"] for r in table)
            style = {"label": label, "linestyle": linestyle}
            # Balanced accuracy against p at the largest ratio.
            at_ratio = [r for r in table if r["ratio"] == ratio_max]
            curve = sweep_curve(at_ratio, N, "p", "balanced_accuracy")
            draw_curve(ax_bal, curve, color, **style)
            # NMSE against the ratio at the largest p.
            at_p = [r for r in table if r["p"] == p_max]
            curve_nmse = sweep_curve(at_p, N, "ratio", "nmse")
            draw_curve(ax_nmse, curve_nmse, color, log=True, **style)
            data[f"{label} N={N}"] = {
                "ratio_max": ratio_max,
                "p_max": p_max,
                "balanced_accuracy_vs_p": [
                    (r["p"], r["balanced_accuracy_mean"]) for r in curve.rows
                ],
                "nmse_vs_ratio": [
                    (r["ratio"], r["nmse_mean"]) for r in curve_nmse.rows
                ],
            }
        ax_bal.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax_bal.set(xlabel="measurement probability p", ylim=(0.4, 1.02))
        ax_bal.set(
            ylabel="balanced accuracy (test)",
            title=f"readout at alpha_ratio = {ratio_max:.3g}, N = {N}",
        )
        ax_nmse.axvline(
            1.0, color="black", linestyle="--", label="interpolation threshold"
        )
        ax_nmse.set_xscale("log")
        ratio_ticks(ax_nmse, [r["ratio"] for r in curves[0][1] if r["N"] == N])
        ax_nmse.set(
            xlabel="alpha_ratio = n_train / n_params",
            ylabel="NMSE (test)",
            title=f"NMSE at p = {p_max:g}, N = {N}",
        )
        for ax in row:
            ax.legend()
    return fig, axes, data


fig_cmp, axes_cmp, comparison = draw_comparison(
    [(model, tables[model], model_colors[model], "-") for model in models]
)
fig_cmp.suptitle("QELM on M: TFIM, Haar and Clifford dynamics")
comparison_path = save_figure(
    fig_cmp,
    sweep_dir,
    "comparison_models",
    data={"comparison": comparison},
)
print(f"figure: {comparison_path}")
plt.show()

## Fixed circuit

With new gates for every sample, the readout of the Haar and Clifford circuits stays at chance. The map from the record to the label changes with the gates, and a linear readout, the same for every sample, has nothing common to learn. The TFIM escapes this: it conserves the parity of the system charge, so the parity of the last outcomes gives the label whatever the disorder.

Here each circuit keeps one draw of its gates for every sample (`<model>.fixed_instance=true`, passed to the launcher through `EXTRA_OVERRIDES`): the map from the record to the label is then the same for every sample. The seed selects the circuit, so the mean over the seeds is a mean over `len(SEEDS)` circuits, and the error bars are the spread between circuits. Same grid, one subfolder `<model>_fixed` per circuit, and the same figures: the curves against the ratio and the heatmaps of the fixed circuits (`metrics_vs_alpha_ratio_fixed.png`, `heatmaps_p_alpha_ratio_fixed.png`), then the comparison with the new-gates sweep, a fixed circuit dashed (`comparison_fixed_circuit.png`). `FIXED = False` skips this section.

In [ ]:
# The circuits of MODELS: the TFIM draws its disorder per sample here.
fixed_models = [m for m in models if m != "TFIM"] if FIXED else []
if RUN:
    for model in fixed_models:
        print(f"model: {model}, fixed circuit")
        launch(model, sweep_dir / f"{model}_fixed", f"{model}.fixed_instance=true")
tables_fixed = {
    model: read_table(model, sweep_dir / f"{model}_fixed", fixed=True)
    for model in fixed_models
}

In [ ]:
if fixed_models:
    fig_fixed, axes_fixed = draw_ratio_curves(tables_fixed)
    fig_fixed.suptitle("QELM on M, fixed circuit: metrics against the ratio")
    figure_fixed_path = save_figure(
        fig_fixed,
        sweep_dir,
        "metrics_vs_alpha_ratio_fixed",
        data={"tables": tables_fixed},
    )
    print(f"figure: {figure_fixed_path}")
    plt.show()

    fig_maps_fixed, axes_maps_fixed, grids_fixed = draw_heatmaps(tables_fixed)
    fig_maps_fixed.suptitle("QELM on M, fixed circuit: metrics over p and the ratio")
    heatmap_fixed_path = save_figure(
        fig_maps_fixed,
        sweep_dir,
        "heatmaps_p_alpha_ratio_fixed",
        data={"tables": tables_fixed},
    )
    print(f"figure: {heatmap_fixed_path}")
    plt.show()

    # New gates per sample solid, a fixed circuit dashed, same color.
    curves = [(model, tables[model], model_colors[model], "-") for model in models]
    curves += [
        (f"{model}, fixed", tables_fixed[model], model_colors[model], "--")
        for model in fixed_models
    ]
    fig_cmp_fixed, axes_cmp_fixed, comparison_fixed = draw_comparison(curves)
    fig_cmp_fixed.suptitle("QELM on M: new gates per sample (solid) or fixed (dashed)")
    comparison_fixed_path = save_figure(
        fig_cmp_fixed,
        sweep_dir,
        "comparison_fixed_circuit",
        data={"comparison": comparison_fixed},
    )
    print(f"figure: {comparison_fixed_path}")
    plt.show()